## I - Importing libraries + Defining parameters and data generation

In [ ]:
import random
import numpy as np
import pandas as pd
from pandas import DataFrame
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import plotly.express as px
from ortools.sat.python import cp_model
from IPython.display import display

class Config:
    T_BASE : int
    P :float
    Q : float
    DELTA_PARALLEL : int
    MULTIPLIERS : dict

def initialize(
        t_base: int, 
        p: float, 
        q: float, 
        delta_parallel: int
):
    """
    Initializes the global parameters for the scheduling. 

    :param t_base: the temporal unit for a 1-story point task's completion.
    :param p: a scale multiplier (<1) for faster execution of tasks.
    :param q: a scale multiplier (>1) for slower execution of tasks.
    :param delta_parallel: the temporal unit for the overlap between two tasks. 
    Used to quantify parallel execution and define overlap constraints.
    """
    Config.T_BASE = t_base
    Config.P = p
    Config.Q = q
    Config.DELTA_PARALLEL = delta_parallel
    Config.MULTIPLIERS = {
        'I': 1, # Intermediate
        'A': p, # Advanced
        'B': q  # Beginner
    }

def generate_dependencies(
        tasks: DataFrame, 
        num_cycles: int, 
) -> list[tuple[str, str, dict[str, str]]]:
    """
    Generates dependencies where tasks heavily favor intramodular links,
    with controlled intermodular cross-links and safe cycles.
    """
    dependencies = []
    task_ids = tasks.index.tolist()
    
    # Track existing source-target pairs to prevent overlap
    existing_pairs = set()
    
    # Group tasks by module for easy lookup
    module_groups = tasks.groupby('module').groups

    # 1. Build dense Intra-Modular Backbones
    for mod, mod_tasks in module_groups.items():
        mod_list = list(mod_tasks)
        for i, child in enumerate(mod_list):
            if i == 0:
                continue
            
            if random.random() < 0.7 and i > 0:
                parent = random.choice(mod_list[:i])
            else:
                parent = random.choice(task_ids)
                if parent == child:
                    parent = mod_list[0]
            
            # Avoid self-loops and duplicate pairs from step 1
            if parent != child and (parent, child) not in existing_pairs:
                dependencies.append((parent, child, {'relation': 'acyc'}))
                existing_pairs.add((parent, child))

    # 2. Inject Controlled Multi-Node Cycles
    cycles_added = 0
    attempts = 0
    while cycles_added < num_cycles and attempts < 100:
        attempts += 1
        
        mod = random.choice(list(module_groups.keys()))
        mod_tasks = list(module_groups[mod])
        
        max_cycle_len = min(5, len(mod_tasks))
        if max_cycle_len < 3:
            continue
            
        cycle_len = random.randint(3, max_cycle_len)
        cycle_nodes = random.sample(mod_tasks, cycle_len)
        
        ring_edges = []
        valid_ring = True
        
        for i in range(cycle_len):
            u = cycle_nodes[i]
            v = cycle_nodes[(i + 1) % cycle_len]
            
            # Prevent self-loops or using a pair already claimed as 'acyc'
            if u == v or (u, v) in existing_pairs:
                valid_ring = False
                break
                
            ring_edges.append((u, v, {'relation': 'cyc'}))
            
        if valid_ring:
            for u, v, attr in ring_edges:
                dependencies.append((u, v, attr))
                existing_pairs.add((u, v))
            cycles_added += 1

    return dependencies

def generate_availabilities(
        max_horizon: int, 
        min_blocks: int, 
        max_blocks: int
) -> list[tuple[int, int]]:
    """
    Generates a list of random, non-overlapping intermittent availability windows.
    Example output: [(12, 180), (250, 410), (600, 850)]
    """
    windows = []
    # Determine how many availability blocks this developer will have
    num_blocks = random.randint(min_blocks, max_blocks)
    
    # Divide the timeline roughly into segments to prevent total overlap chaos
    segment_size = max_horizon // (num_blocks + 1)
    
    for i in range(num_blocks):
        # Define a safe lower and upper bound for this specific block
        start_min = i * segment_size + random.randint(0, 20)
        start_max = start_min + (segment_size // 2)
        
        start = random.randint(min(start_min, max_horizon - 50), min(start_max, max_horizon - 30))
        duration = random.randint(50, segment_size)
        finish = min(start + duration, max_horizon)
        
        if start < finish:
            windows.append((start, finish))
            
    # Sort windows sequentially just in case
    windows.sort(key=lambda x: x[0])
    return windows

def generate_project_data(
        num_tasks: int, 
        num_devs: int, 
        horizon: int, 
        seed=40,
        min_blocks_avail=2, 
        max_blocks_avail=3,
        num_cycle=3,
        toggle_display=False
) -> tuple[DataFrame, DataFrame, list[tuple[str, str, dict[str, str]]]]:
    """
    Generates a solvable, realistic project dataset with a valid DAG 
    and complete profile coverage.
    """
    random.seed(seed)
    
    # 1. Define standard types, modules, and team mappings
    all_types = ['database', 'backend', 'frontend', 'testing', 'devops']
    all_modules = ['data', 'auth', 'api', 'ui', 'dashboard', 'qa', 'infra']
    module_to_team = {
        'data': 'team_B', 
        'auth': 'team_A', 
        'api': 'team_A', 
        'ui': 'team_A', 
        'dashboard': 'team_A', 
        'qa': 'team_B', 
        'infra': 'team_B'
    }
    
    # 2. Generate Tasks
    task_ids = [f'T{i}' for i in range(1, num_tasks + 1)]
    sp_list = [random.choice([1, 1, 2, 3, 5, 8, 13]) for _ in range(num_tasks)]
    type_list = [random.choice(all_types) for _ in range(num_tasks)]
    module_list = [random.choice(all_modules) for _ in range(num_tasks)]
    
    tasks = DataFrame({
        'task_id': task_ids,
        'sp': sp_list,
        'type': type_list,
        'module': module_list
    }).set_index('task_id')
    
    tasks['team_required'] = tasks['module'].map(module_to_team)
    
    # 3. Generate Developers (Guaranteeing full profile coverage for all task modules/types)
    dev_ids = [f'D{i}' for i in range(1, num_devs + 1)]
    profiles = []
    teams = []
    exps = []
    availabilities = []

    # Ensure developers collectively cover every required type/module
    universal_profile = list(set(all_types))
    for i in range(num_devs):
        if i == 0:
            # First dev is a senior full-stack expert covering everything to prevent bottlenecks
            profiles.append(universal_profile)
            teams.append('team_A')
            exps.append('A')
        else:
            profiles.append(random.sample(all_types, k=min(3, len(all_types))))
            teams.append(random.choice(['team_A', 'team_B']))
            exps.append(random.choice(['B', 'I', 'A']))
        availabilities.append(generate_availabilities(horizon, min_blocks_avail, max_blocks_avail))
        
    developers = DataFrame({
        'dev_id': dev_ids,
        'profile': profiles,
        'team': teams,
        'exp': exps,
        'availability': availabilities
    }).set_index('dev_id')
    
    # 4. Generate Safe, Strictly Feed-Forward Dependencies (Valid DAG)
    # This guarantees no deadlocks or impossible structural loops.
    dependencies = generate_dependencies(tasks, num_cycle)

    if toggle_display:
        display(tasks)
        display(developers)
        display_dependencies(dependencies)
        display_dependency_graph(tasks, dependencies)
        display_module_dependency_sheet(tasks, dependencies)
        display_dev_times_matrix(tasks, developers)
        display_availability_windows(developers)
        
    return tasks, developers, dependencies

## II - Building helper functions

### II.1 Dependency Graph

In [ ]:
def build_dependency_graph(
        tasks: DataFrame, 
        dependencies: list[tuple[str, str, dict[str, str]]]
) -> nx.DiGraph:
    """
    Builds the dependency graph relative to the given tasks and dependencies.
    """
    G = nx.DiGraph()
    G_acyc = nx.DiGraph()

    task_data = tasks.to_dict('index')
    for task_id, data in task_data.items():
        G.add_node(task_id, **data)
        G_acyc.add_node(task_id, **data)

    for u, v, attr in dependencies:
        G.add_edge(u, v, **attr)
        if attr.get('relation') == 'acyc':
            G_acyc.add_edge(u, v)
        elif attr.get('relation') == 'cyc':
            G.add_edge(v, u, **attr)  # reciprocal for cycle detection only

    for node in G.nodes():
        G.nodes[node]['dep_count'] = len(nx.descendants(G_acyc, node))

    return G

### II.2 Feasibility

In [ ]:
def check_feasibility(
        tasks: DataFrame, 
        developers: DataFrame, 
        task_id: str, 
        dev_id: str
) -> bool:
    """
    Checks whether the developer (dev_id) is eligible for the giving task (task_id).
    """
    task = tasks.loc[task_id]
    dev = developers.loc[dev_id]
    
    # Rule 1: Skill Matching
    skill_match = task['type'] in dev['profile']

    # Rule 2: Team-Module Ownership
    team_match = dev['team'] == task['team_required']
    
    return skill_match and team_match

### II.3 Development Times

In [ ]:
def a_dev_time(
        tasks: DataFrame, 
        developers: DataFrame, 
        task_id: str, 
        dev_id: str
) -> float:
    """
    Calculates the average development time of the task (task_id) by the dev (dev_id).
    """
    # dev_time(t_i, d_j) = sp(t_i) * T_{base} * beta(exp(d_j))
    task = tasks.loc[task_id]
    dev = developers.loc[dev_id]
    exp = dev['exp']

    return task['sp'] * Config.T_BASE * Config.MULTIPLIERS.get(exp)

def build_dev_times_matrix(
        tasks: DataFrame, 
        developers: DataFrame
) -> DataFrame:
    """
    Builds a matrix where rows are tasks and columns are developers,
    containing the calculated development time for each pair.
    """
    matrix_data = []

    for task_id in tasks.index:
        row = []
        for dev_id in developers.index:
            duration = a_dev_time(tasks, developers, task_id, dev_id)
            row.append(duration)
        matrix_data.append(row)

    return pd.DataFrame(
        matrix_data, 
        index=tasks.index, 
        columns=developers.index
    )

### II.4 Displayers

In [ ]:
def display_dev_times_matrix(
        tasks: DataFrame, 
        developers: DataFrame
):
    """
    Displays the development times matrix relative to the tasks and developers.
    """
    dev_times = build_dev_times_matrix(tasks, developers)
    print("Development Times Matrix:")
    display(dev_times)

def display_dependencies(dependencies: list[tuple[str, str, dict[str, str]]]):
    """
    Displays the dependencies in a tabular view.
    """
    dependencies_df = DataFrame(
    [((tu, tv), d["relation"]) for tu, tv, d in dependencies], columns=["dep", "relation"]
    ).set_index("dep")
    display(dependencies_df)

def display_availability_windows(developers: DataFrame):
    """
    Displays developers availability windows as a Gantt-style chart.
    """
    # 1. Unpack the availability intervals from the DataFrame into a flat plotting structure
    records = []
    for dev_id, row in developers.iterrows():
        avail_windows = row['availability']
        for i, (start, finish) in enumerate(avail_windows):
            records.append({
                'Developer': dev_id,
                'Window_ID': f"Window {i+1}",
                'Start': start,
                'Finish': finish
            })
            
    plot_df = pd.DataFrame(records)
    
    # If no availability data is present, handle gracefully
    if plot_df.empty:
        print("No availability windows to plot.")
        return

    plot_df['Duration'] = plot_df['Finish'] - plot_df['Start'] 
    
    # 2. Plot using Plotly Express with the requested style format
    fig = px.bar(
        plot_df, 
        x="Duration",
        y="Developer", 
        color="Developer", 
        orientation='h', 
        base="Start",
        custom_data=["Start", "Finish", "Duration"], 
        title="Developer Availability Windows Over Time"
    )
    
    # 3. Custom hover template matching your style structure
    fig.update_traces(
        hovertemplate=(
            "Developer: %{y}<br>"
            "Start: %{customdata[0]}<br>"
            "Finish: %{customdata[1]}<br>"
            "Duration: %{customdata[2]}<extra></extra>"
        )
    )
    
    fig.update_layout(xaxis_title="Time Units", yaxis_title="Developers")
    fig.update_yaxes(autorange="reversed")
    
    fig.show()

def display_dependency_graph(
        tasks: DataFrame, 
        dependencies: list[tuple[str, str, dict[str, str]]]
):
    """
    Displays tasks grouped into their respective modules, crisscrossed by depedency links.
    """
    G = nx.DiGraph()
    
    # Add nodes with module attributes
    for t_id, row in tasks.iterrows():
        G.add_node(t_id, module=row['module'], type=row['type'])
        
    # Add edges with relation attributes
    for u, v, data in dependencies:
        G.add_edge(u, v, relation=data.get('relation', 'acyc'))

    fig, ax = plt.subplots(figsize=(10, 10))
    
    # 1. Group tasks by module
    modules = tasks['module'].unique()
    num_modules = len(modules)
    
    # Compute macro-centers for each module pod in a circle
    pod_centers = {}
    radius_macro = 4.5
    for i, mod in enumerate(modules):
        angle = 2 * np.pi * i / num_modules
        pod_centers[mod] = np.array([radius_macro * np.cos(angle), radius_macro * np.sin(angle)])
        
    # 2. Compute local micro-positions for tasks inside their respective pods uniformly
    pos = {}
    pod_radii = {}
    
    for mod in modules:
        mod_tasks = [t for t, data in G.nodes(data=True) if data['module'] == mod]
        num_mod_tasks = len(mod_tasks)
        
        # Give the pod a fixed radius big enough to hold its tasks cleanly
        # e.g., scale radius based on the number of tasks in this module
        base_radius = max(1.2, 0.5 * np.sqrt(num_mod_tasks) + 0.8)
        pod_radii[mod] = base_radius
        
        # Arrange tasks regularly in a circle (or concentric circles if too many) inside the pod
        for idx, t in enumerate(mod_tasks):
            if num_mod_tasks == 1:
                # Single task sits right at the pod center
                pos[t] = pod_centers[mod]
            else:
                # Distribute evenly along the perimeter of an inner sub-radius
                inner_radius = base_radius * 0.8
                angle = 2 * np.pi * idx / num_mod_tasks
                offset = np.array([inner_radius * np.cos(angle), inner_radius * np.sin(angle)])
                pos[t] = pod_centers[mod] + offset

    # 3. Draw Clean Circular Module Pod Backgrounds
    color_palette = ['#ff9999', '#99ff99', '#9999ff', '#ffcc99', '#ffff99']
    for i, mod in enumerate(modules):
        color = color_palette[i % len(color_palette)]
        
        # Using Circle patch with equal aspect ratio ensured by ax.set_aspect('equal')
        circle = patches.Circle(
            pod_centers[mod], pod_radii[mod], 
            color=color, alpha=0.3, zorder=0
        )
        ax.add_patch(circle)
        
        # Add Module Label Header at the top-center of each pod circle
        ax.text(
            pod_centers[mod][0], pod_centers[mod][1] + pod_radii[mod] + 0.2, 
            f"Module: {mod}", fontweight='bold', fontsize=10, 
            ha='center', va='center', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
        )

    # 4. Categorize Edges for Styling
    intra_edges = []
    inter_edges = []
    cyclic_edges = []
    
    cyclic_pairs = set()
    for u, v, data in G.edges(data=True):
        if data.get('relation') == 'cyc' or G.has_edge(v, u):
            cyclic_pairs.add(tuple(sorted((u, v))))

    seen_cycles = set()  # Tracks pairs already added to avoid duplicates

    for u, v, data in G.edges(data=True):
        u_mod = G.nodes[u]['module']
        v_mod = G.nodes[v]['module']
        pair = tuple(sorted((u, v)))
        
        if pair in cyclic_pairs:
            if pair not in seen_cycles:
                cyclic_edges.append((u, v))
                seen_cycles.add(pair)
        elif u_mod == v_mod:
            intra_edges.append((u, v))
        else:
            inter_edges.append((u, v))

    # 5. Render Nodes & Labels
    nx.draw_networkx_nodes(G, pos, node_size=400, node_color='white', edgecolors='black', ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)

    # 6. Render Edges with Custom Curves & Colors
    # Intra-module (Gray, subtle inside pods)
    nx.draw_networkx_edges(G, pos, edgelist=intra_edges, edge_color='gray', arrows=True, width=1, ax=ax)
    
    # Inter-module (Green, curved to bypass pods cleanly)
    nx.draw_networkx_edges(
        G, pos, edgelist=inter_edges, edge_color='green', 
        arrows=True, width=1, connectionstyle='arc3,rad=0.2', ax=ax
    )
    
    # Cyclic Dependencies (Red, prominent curved arrows)
    nx.draw_networkx_edges(
        G, pos, edgelist=cyclic_edges, edge_color='red', 
        arrows=True, arrowstyle='<->', width=1, 
        connectionstyle='arc3,rad=0.3', ax=ax
    )

    # Ensure circular pods do not distort into ovals when scaling canvas
    ax.set_aspect('equal')
    ax.set_axis_off()
    plt.suptitle("Dependency Graph", fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

def display_module_dependency_sheet(
        tasks: DataFrame, 
        dependencies: list[tuple[str, str, dict[str, str]]]
):
    """
    Displays all module sub-graphs with the according tasks and dependency links,
    arranged onto a single multi-panel sheet.
    """
    # 1. Build the full graph
    G = nx.DiGraph()
    for t_id, row in tasks.iterrows():
        G.add_node(t_id, module=row['module'], type=row['type'])
        
    for u, v, data in dependencies:
        G.add_edge(u, v, relation=data.get('relation', 'acyc'))

    modules = tasks['module'].unique()
    num_modules = len(modules)
    color_palette = ['#ff9999', '#99ff99', '#9999ff', '#ffcc99', '#ffff99']

    # Detect global cyclic pairs
    cyclic_pairs = set()
    for u, v, data in G.edges(data=True):
        if data.get('relation') == 'cyc' or G.has_edge(v, u):
            cyclic_pairs.add(tuple(sorted((u, v))))

    # 2. Setup grid layout
    ncols = 2
    nrows = (num_modules + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(12, 6 * nrows))
    axes = np.array(axes).flatten()

    # Find global max task count to establish a uniform frame limit for perfect circles
    max_mod_tasks = max(len([t for t, d in G.nodes(data=True) if d['module'] == m]) for m in modules)
    global_max_radius = 1.2 + 0.6 * np.sqrt(max_mod_tasks) + 2.5

    # 3. Loop through each module
    for i, target_mod in enumerate(modules):
        ax = axes[i]
        
        internal_tasks = [t for t, data in G.nodes(data=True) if data['module'] == target_mod]
        
        external_tasks = set()
        for u, v in G.edges():
            if u in internal_tasks and v not in internal_tasks:
                external_tasks.add(v)
            elif v in internal_tasks and u not in internal_tasks:
                external_tasks.add(u)
        external_tasks = list(external_tasks)

        # 4. Proportional Scaling based on a Fixed Minimum Radius
        num_internal = len(internal_tasks)
        base_radius = max(1.4, 0.6 * np.sqrt(num_internal) + 0.8)
        
        # Position internal tasks inside the pod
        pos = {}
        for idx, t in enumerate(internal_tasks):
            if num_internal == 1:
                pos[t] = np.array([0.0, 0.0])
            else:
                inner_radius = base_radius * 0.8
                angle = 2 * np.pi * idx / num_internal
                pos[t] = np.array([inner_radius * np.cos(angle), inner_radius * np.sin(angle)])

        # Position external boundary tasks on an outer ring
        outer_radius = base_radius + 2.0
        num_external = len(external_tasks)
        for idx, t in enumerate(external_tasks):
            angle = 2 * np.pi * idx / max(1, num_external)
            pos[t] = np.array([outer_radius * np.cos(angle), outer_radius * np.sin(angle)])

        # 5. Draw Pod Background (Header now displays ONLY the module title)
        color = color_palette[i % len(color_palette)]
        circle = patches.Circle((0, 0), base_radius, color=color, alpha=0.3, zorder=0)
        ax.add_patch(circle)
        
        ax.text(
            0, base_radius + 0.35, 
            f"Module: {target_mod}", fontweight='bold', fontsize=10, 
            ha='center', va='center', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray')
        )

        # 6. Categorize Edges (Avoiding duplicate lines for mutual cycles)
        intra_edges, inter_edges, cyclic_edges = [], [], []
        drawn_pairs = set()

        for u, v in G.edges():
            if u in internal_tasks or v in internal_tasks:
                pair = tuple(sorted((u, v)))
                if pair in cyclic_pairs:
                    if pair not in drawn_pairs:
                        cyclic_edges.append((u, v))
                        drawn_pairs.add(pair)
                elif u in internal_tasks and v in internal_tasks:
                    intra_edges.append((u, v))
                else:
                    inter_edges.append((u, v))

        # 7. Render Nodes & Labels
        all_displayed_nodes = internal_tasks + external_tasks
        nx.draw_networkx_nodes(G, pos, nodelist=internal_tasks, node_size=400, node_color='white', edgecolors='black', ax=ax)
        if external_tasks:
            nx.draw_networkx_nodes(G, pos, nodelist=external_tasks, node_size=300, node_color='#e0e0e0', edgecolors='gray', ax=ax)
            
        nx.draw_networkx_labels(G, pos, labels={n: n for n in all_displayed_nodes if n in pos}, font_size=8, font_weight='bold', ax=ax)

        # 8. Render Edges
        nx.draw_networkx_edges(G, pos, edgelist=intra_edges, edge_color='gray', arrows=True, width=1, ax=ax)
        nx.draw_networkx_edges(G, pos, edgelist=inter_edges, edge_color='green', arrows=True, width=1, connectionstyle='arc3,rad=0.2', ax=ax)
        nx.draw_networkx_edges(G, pos, edgelist=cyclic_edges, edge_color='red', arrows=True, arrowstyle='<->', width=1, connectionstyle='arc3,rad=0.3', ax=ax)

        # Lock axis boundaries uniformly to maintain perfect circles
        limit = global_max_radius
        ax.set_xlim(-limit, limit)
        ax.set_ylim(-limit, limit)
        ax.set_aspect('equal')
        ax.set_axis_off()

    # Hide unused panels
    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle("Module Dependency Sheet", fontsize=14, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()
    
def display_gantt_chart(schedule):
    """
    Displays the schedule as a Gantt-style chart.
    """
    if isinstance(schedule, dict):
        plot_df = DataFrame.from_dict(schedule, orient='index', columns=['assigned_dev', 'start', 'finish'])
    else:
        plot_df = schedule.copy()

    plot_df = plot_df.reset_index()
    plot_df.columns = ['Task', 'Developer', 'Start', 'Finish']
    
    plot_df['Duration'] = plot_df['Finish'] - plot_df['Start'] 
    
    fig = px.bar(
        plot_df, 
        x="Duration",
        y="Task", 
        color="Developer", 
        orientation='h', 
        base="Start",
        custom_data=["Developer", "Start", "Finish", "Duration"], 
        title="Project Schedule"
    )
    
    fig.update_traces(
        hovertemplate=(
            "Task: %{y}<br>"
            "Developer: %{customdata[0]}<br>"
            "Start: %{customdata[1]}<br>"
            "Finish: %{customdata[2]}<br>"
            "Duration: %{customdata[3]}<extra></extra>"
        )
    )
    
    fig.update_layout(xaxis_title="Time Units", yaxis_title="Tasks")
    fig.update_yaxes(autorange="reversed")
    fig.show()

## III - Objective Functions & Constraints

In [ ]:
def calculate_makespan(schedule) -> int:
    """
    Calculates the makespan of the project schedule.
    """
    # Safely convert to a DataFrame if it's passed as a dictionary
    if isinstance(schedule, dict):
        schedule_df = DataFrame.from_dict(schedule, orient='index', columns=['assigned_dev', 'start', 'finish'])
    elif isinstance(schedule, DataFrame):
        schedule_df = schedule
    else:
        # Fallback conversion or handle array inputs safely
        schedule_df = DataFrame(schedule, columns=['assigned_dev', 'start', 'finish'])
    # Makespan is simply the maximum finish time
    return schedule_df['finish'].max()

def calculate_workload_imbalance(
        schedule, 
        developers: DataFrame
) -> float:
    """
    Calculates the workload imbalance of the schedule based on the workloads of developers.
    """
    # Safely convert to a DataFrame if it's passed as a dictionary
    if isinstance(schedule, dict):
        schedule_df = DataFrame.from_dict(schedule, orient='index', columns=['assigned_dev', 'start', 'finish'])
    elif isinstance(schedule, DataFrame):
        schedule_df = schedule
    else:
        # Fallback conversion or handle array inputs safely
        schedule_df = DataFrame(schedule, columns=['assigned_dev', 'start', 'finish'])

    workloads = []
    for dev_id in developers.index:
        dev_tasks = schedule_df[schedule_df['assigned_dev'] == dev_id]
        w_j = (dev_tasks['finish'] - dev_tasks['start']).sum()
        workloads.append(w_j)
        
    return np.std(workloads)

def calculate_coordination_risk(u, v, su, fu, sv, fv, G: nx.DiGraph) -> int:
    """
    Calculates the Coordination Risk (CR) relative to the acyclic dependency between task_u and task_v.
    """
    # Sequential condition
    if (fu - sv) < Config.DELTA_PARALLEL:
        return 0

    # Parallel condition
    overlap = min(fu, fv) - max(su, sv)
    if overlap >= Config.DELTA_PARALLEL:
        return 1 + G.nodes[v].get('dep_count', 0)
    
    return 0

def calculate_total_coordination_risk(
        schedule,
        G: nx.DiGraph, 
        dependencies: list[tuple[str, str, dict[str, str]]]
) -> int:
    """
    Calculates the total coordination risk of the schedule relative to the dependency graph.
    """
    # Safely convert to a DataFrame if it's passed as a dictionary
    if isinstance(schedule, dict):
        schedule_df = DataFrame.from_dict(schedule, orient='index', columns=['assigned_dev', 'start', 'finish'])
    elif isinstance(schedule, DataFrame):
        schedule_df = schedule
    else:
        # Fallback conversion or handle array inputs safely
        schedule_df = DataFrame(schedule, columns=['assigned_dev', 'start', 'finish'])

    total_risk = 0
    
    # 1. Sum Acyclic Risks (P_acyc)
    # Filter edges that are NOT part of any cycle
    for u, v, data in G.edges(data=True):
        if data.get('relation') == 'acyc':
            su, fu = schedule_df.loc[u, 'start'], schedule_df.loc[u, 'finish']
            sv, fv = schedule_df.loc[v, 'start'], schedule_df.loc[v, 'finish']
            total_risk += calculate_coordination_risk(u, v, su, fu, sv, fv, G)

    # 2. Sum Cyclic Risks (P_cyc)
    # Filter dependencies for cyclic edges
    cyc_edges = [(u, v) for u, v, attr in dependencies if attr.get('relation') == 'cyc']

    # Eq. (9): CRC = sum over edges in cycle of (1 + dep_count(t_k))
    for u, v in cyc_edges:
        target_risk = 1 + G.nodes[v].get('dep_count', 0)
        total_risk += target_risk
        
    return total_risk

def check_constraints(
        schedule, 
        tasks: DataFrame, 
        developers: DataFrame, 
        dependencies: list[tuple[str, str, dict[str, str]]]
) -> bool:
    """
    Checks whether the schedule validates all constraints and flags violated ones. \\
    Returns True if all checks pass, False otherwise. 
    """
    # Safely convert to a DataFrame if it's passed as a dictionary
    if isinstance(schedule, dict):
        schedule_df = DataFrame.from_dict(schedule, orient='index', columns=['assigned_dev', 'start', 'finish'])
    elif isinstance(schedule, DataFrame):
        schedule_df = schedule
    else:
        # Fallback conversion or handle array inputs safely
        schedule_df = DataFrame(schedule, columns=['assigned_dev', 'start', 'finish'])

    # 1. Skill & Team Constraints
    for task_id, row in schedule_df.iterrows():
        dev_id = row['assigned_dev']
        if not check_feasibility(tasks, developers, task_id, dev_id):
            print(f"Constraint Violated: Skill/Team mismatch for {task_id} assigned to {dev_id}.")
            return False

    # 2. Availability Constraint
    for task_id, row in schedule_df.iterrows():
        dev_id = row['assigned_dev']
        s_i, f_i = row['start'], row['finish']
        avail_dev = developers.loc[dev_id, 'availability']
        is_available = False
        for slot_start, slot_end in avail_dev:
            if s_i >= slot_start and f_i <= slot_end:
                is_available = True
                break
        if not is_available:
            print(f"Constraint Violated: {task_id} interval [{s_i}, {f_i}] outside {dev_id} availability {avail_dev}.")
            return False

    # 3. Flow Constraints (Acyclic & Cyclic)
    for u, v, attr in dependencies:
        su, fu = schedule_df.loc[u, 'start'], schedule_df.loc[u, 'finish']
        sv, fv = schedule_df.loc[v, 'start'], schedule_df.loc[v, 'finish']
        
        if attr['relation'] == 'acyc':
            if fv - su < Config.DELTA_PARALLEL:   # f_v - s_u >= delta
                print(f"Constraint Violated: Acyclic dependency {u}->{v} direction not respected.")
                return False
        elif attr['relation'] == 'cyc':
            overlap = min(fu, fv) - max(su, sv)
            if overlap < Config.DELTA_PARALLEL:
                print(f"Constraint Violated: Cyclic dependency {u} <-> {v} lacks required overlap.")
                return False

    # 4. Resource Constraint (No overlapping tasks for the same dev)
    for dev_id in schedule_df['assigned_dev'].unique():
        dev_tasks = schedule_df[schedule_df['assigned_dev'] == dev_id].sort_values('start')
        for i in range(len(dev_tasks) - 1):
            if dev_tasks.iloc[i+1]['start'] < dev_tasks.iloc[i]['finish']:
                print(f"Constraint Violated: Resource conflict for {dev_id} on {dev_tasks.iloc[i].name} and {dev_tasks.iloc[i+1].name}.")
                return False 

    return True

## IV - Building the schedulers and the test-runner

### IV.1 CP-SAT Scheduler

In [ ]:
def cp_sat_model(
        tasks: DataFrame, 
        developers: DataFrame, 
        dependencies: list[tuple[str, str, dict[str, str]]], 
        horizon=1000
) -> tuple[cp_model.CpModel, dict]:
    """
    Builds the CP-SAT model relative to the given data. \\
    Returns the model and its variables.
    """
    model = cp_model.CpModel()
    
    dev_time_matrix = build_dev_times_matrix(tasks, developers)
    G = build_dependency_graph(tasks, dependencies)
    
    # 1. Variables
    starts = {t: model.new_int_var(0, horizon, f'start_{t}') for t in tasks.index}
    ends = {t: model.new_int_var(0, horizon, f'end_{t}') for t in tasks.index}
    assignments = {(t, d): model.new_bool_var(f'assign_{t}_{d}') for t in tasks.index for d in developers.index}
    makespan = model.new_int_var(0, horizon, 'makespan')
    imbalance = model.new_int_var(0, horizon, 'imbalance')
    total_risk = model.new_int_var(0, horizon * max(1, len(dependencies)), 'total_risk')
    
    # 2. Constraints

    # Task Assignment & Feasibility
    for t in tasks.index:
        # Each task must be assigned exactly once
        model.add(sum(assignments[(t, d)] for d in developers.index) == 1)
        
        # Skill & Team constraints: forbid invalid assignments
        for d in developers.index:
            if not check_feasibility(tasks, developers, t, d):
                model.add(assignments[(t, d)] == 0)

    # Availability
    for t in tasks.index:
        for d in developers.index:
            # Get intervals for this specific dev
            avail_intervals = developers.loc[d, 'availability']
            fits_in_window = []
            for i, (s_avail, e_avail) in enumerate(avail_intervals):
                fits = model.new_bool_var(f'fits_{t}_{d}_{i}')
                model.add(starts[t] >= s_avail).only_enforce_if(fits)
                model.add(ends[t] <= e_avail).only_enforce_if(fits)
                fits_in_window.append(fits)
            
            # If assigned, it MUST fit
            model.add(sum(fits_in_window) >= 1).only_enforce_if(assignments[(t, d)])

    # Duration & Temporal Linking
    chosen_durations = {}

    for t in tasks.index:
        # Get the list of times for this task across all developers
        dev_times = [int(dev_time_matrix.loc[t, d]) for d in developers.index]
        
        # Create the duration variable for the task
        chosen_durations[t] = model.new_int_var(0, horizon, f'dur_{t}')
        
        # Link the chosen developer's time to the duration variable
        for i, d in enumerate(developers.index):
            model.add(chosen_durations[t] == dev_times[i]).only_enforce_if(assignments[(t, d)])
            
        # Link start, end, and duration
        model.add(ends[t] == starts[t] + chosen_durations[t])

    # Project Dependencies
    for u, v, attr in dependencies:
        if attr['relation'] == 'acyc':
            model.add(ends[v] - starts[u] >= Config.DELTA_PARALLEL)
            
        elif attr['relation'] == 'cyc':
            # Logic: Enforce minimum overlap (DELTA_PARALLEL)
            overlap_start = model.new_int_var(0, horizon, f'overlap_start_{u}_{v}')
            overlap_end = model.new_int_var(0, horizon, f'overlap_end_{u}_{v}')
            
            model.add_max_equality(overlap_start, [starts[u], starts[v]])
            model.add_min_equality(overlap_end, [ends[u], ends[v]])
            
            model.add(overlap_end - overlap_start >= Config.DELTA_PARALLEL)

    # Resource Capacity (No Overlap)
    for d in developers.index:
        # Create interval list per developer for resource scheduling
        intervals = [
            model.new_optional_interval_var(
                starts[t], int(a_dev_time(tasks, developers, t, d)), ends[t], 
                assignments[(t, d)], f'opt_int_{t}_{d}'
            ) for t in tasks.index
        ]
        # Ensure one developer cannot work on two tasks at once
        model.add_no_overlap(intervals)

    # 3. Objective Functions

    # Makespan
    model.add_max_equality(makespan, [ends[t] for t in tasks.index])

    # Workload Imbalance (Range as a proxy for standard deviation)
    # Note: CP-SAT doesn't do floating point division well. 
    # Use total sum to represent "average".
    total_workload = model.new_int_var(0, horizon * len(tasks), 'total_workload')
    model.add(total_workload == sum(chosen_durations[t] for t in tasks.index))

    max_dev_load = model.new_int_var(0, horizon, 'max_load')
    min_dev_load = model.new_int_var(0, horizon, 'min_load')
    dev_loads = []
    for d in developers.index:
        load = model.new_int_var(0, horizon, f'load_{d}')
        model.add(load == sum(assignments[(t, d)] * int(a_dev_time(tasks, developers, t, d)) for t in tasks.index))
        dev_loads.append(load)

    model.add_max_equality(max_dev_load, dev_loads)
    model.add_min_equality(min_dev_load, dev_loads)
    model.add(imbalance == max_dev_load - min_dev_load)

    # Coordination Risk
    risk_terms = []
    
    for u, v, attr in dependencies:
        if attr['relation'] == 'acyc':
            # Create a boolean variable indicating whether tasks u and v overlap by at least DELTA_PARALLEL
            is_parallel = model.new_bool_var(f'parallel_{u}_{v}')
            
            overlap_start = model.new_int_var(0, horizon, f'overlap_start_{u}_{v}')
            overlap_end = model.new_int_var(0, horizon, f'overlap_end_{u}_{v}')
            
            model.add_max_equality(overlap_start, [starts[u], starts[v]])
            model.add_min_equality(overlap_end, [ends[u], ends[v]])
            
            # Link boolean indicator to the overlap condition
            model.add(overlap_end - overlap_start >= Config.DELTA_PARALLEL).only_enforce_if(is_parallel)
            model.add(overlap_end - overlap_start < Config.DELTA_PARALLEL).only_enforce_if(is_parallel.Not())
            
            descendant_count = len(list(nx.descendants(G, v)))
            risk_weight = 1 + descendant_count
            
            # Create an integer term for this edge's risk contribution
            edge_risk = model.new_int_var(0, horizon, f'risk_{u}_{v}')
            model.add_multiplication_equality(edge_risk, [is_parallel, risk_weight])
            risk_terms.append(edge_risk)
            
        elif attr.get('relation') == 'cyc':
            # Cyclic risk : Sum over directed cycle edges of (1 + dep_count(target))
            risk_terms.append(1 + G.nodes[v].get('dep_count', 0))

    if risk_terms:
        model.add(total_risk == sum(risk_terms))
    else:
        model.add(total_risk == 0)

    variables = {
        'starts': starts,
        'ends': ends,
        'assignments': assignments,
        'makespan': makespan,
        'imbalance': imbalance,
        'total_risk': total_risk
    }
    return model, variables

def cp_sat_weighted_sum_scheduler(
        tasks: DataFrame, 
        developers: DataFrame, 
        dependencies: list[tuple[str, str, dict[str, str]]], 
        horizon=1000, 
        w_makespan=1, 
        w_risk=1, 
        w_imbalance=1
) -> dict:
    """
    Solves the CP-SAT scheduling model using a weighted sum of the objective functions.
    """
    model, variables = cp_sat_model(tasks, developers, dependencies, horizon=horizon)
    makespan, total_risk, imbalance = variables['makespan'], variables['total_risk'], variables['imbalance']
    model.minimize(w_makespan * makespan + w_risk * total_risk + w_imbalance * imbalance)
    solver = cp_model.CpSolver()
    status = solver.solve(model)
    
    return extract_solution(solver, status, tasks, developers, variables)

def cp_sat_lexicographic_scheduler(
        tasks: DataFrame, 
        developers: DataFrame, 
        dependencies: list[tuple[str, str, dict[str, str]]], 
        horizon=1000, 
        priority_order=['makespan', 'total_risk', 'imbalance']
) -> dict:
    """
    Solves the CP-SAT scheduling model lexicographically based on a dynamic priority list.
    
    :param priority_order: a list of objective names in order of importance, 
                           e.g., ['makespan', 'total_risk', 'imbalance']
    """
    locked_bounds = {}
    last_solver = None
    last_status = None
    last_vars = None

    for _, obj_name in enumerate(priority_order):
        # 1. Build a fresh model instance for this phase
        model, variables = cp_sat_model(tasks, developers, dependencies, horizon=horizon)
        
        # 2. Apply hard constraints for all previously optimized objectives
        for prev_obj, optimal_val in locked_bounds.items():
            model.add(variables[prev_obj] <= optimal_val)
            
        # 3. Set the current objective to minimize
        model.minimize(variables[obj_name])
        
        # 4. Solve this phase
        solver = cp_model.CpSolver()
        status = solver.solve(model)
        if status not in [cp_model.OPTIMAL, cp_model.FEASIBLE]:
            print(f"Phase failed: No feasible solution found while optimizing '{obj_name}'.")
            return None
            
        # 5. Capture the optimal value for this objective to lock it in future phases
        optimal_val = solver.value(variables[obj_name])
        locked_bounds[obj_name] = optimal_val
        
        # Keep track of the final solver state
        last_solver = solver
        last_status = status
        last_vars = variables

    # 6. Extract and return the schedule from the final completed phase
    return extract_solution(last_solver, last_status, tasks, developers, last_vars)

def extract_solution(
        solver: cp_model.CpSolver, 
        status: cp_model.CpSolverStatus, 
        tasks: DataFrame, 
        developers: DataFrame, 
        variables: dict
) -> dict:
    """
    Extract the solution from the solver's output. Returns the schedule.
    """
    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        result = {}
        assignments, starts, ends = variables['assignments'], variables['starts'], variables['ends']
        for t in tasks.index:
            assigned_dev = next(d for d in developers.index if solver.value(assignments[(t, d)]) == 1)
            result[t] = [
                assigned_dev, 
                solver.value(starts[t]), 
                solver.value(ends[t])
            ]
        print(f"{status} solution found.")
        return result
    else:
        print("No feasible solution found.")
        return None

### IV.2 PSO-GA Scheduler

In [ ]:
class HybridSchedulerPSOGA:
    def __init__(
            self,
            tasks: DataFrame,
            developers: DataFrame,
            dependencies: list[tuple[str, str, dict[str, str]]],
            swarm_size=50,
            max_iter=100
    ):
        self.tasks = tasks
        self.developers = developers
        self.dependencies = dependencies

        self.swarm_size = swarm_size
        self.max_iter = max_iter

        self.task_list = tasks.index.to_list()
        self.dev_list = developers.index.to_list()

        self.G = build_dependency_graph(tasks, dependencies)

        # Optimization tracking
        self.global_best_position = None
        self.global_best_score = float('inf')
        self.global_best_schedule = None

    def initialize_swarm(self):
        """
        Initializes the swarm with continuous priority weight vectors.
        """
        swarm = []
        for _ in range(self.swarm_size):
            particle = {
                "position": np.random.uniform(0.0, 1.0, len(self.task_list)),
                "velocity": np.random.uniform(-0.1, 0.1, len(self.task_list)),
                "pbest_position": None,
                "pbest_score": float('inf'),
                "pbest_schedule": None
            }
            swarm.append(particle)
        return swarm

    def crossover(self, pos1, pos2):
        """
        Performs blend crossover on continuous priority weight vectors.
        """
        alpha = random.random()
        child_position = alpha * pos1 + (1 - alpha) * pos2
        return child_position

    def mutate(self, position, mutation_rate=0.15):
        """
        Mutates continuous priority weights to inject swarm diversity.
        """
        child_position = position.copy()
        for i in range(len(child_position)):
            if random.random() < mutation_rate:
                child_position[i] = random.uniform(0.0, 1.0)
        return child_position

    #def decode_schedule(self, position): -> schedule

    def evaluate_objectives(self, schedule):
        """
        Evaluates the objective vector: Makespan, Coordination Risk, Workload Imbalance.
        """
        schedule_df = pd.DataFrame.from_dict(
            schedule,
            orient='index',
            columns=['assigned_dev', 'start', 'finish']
        )

        makespan = calculate_makespan(schedule_df)
        total_risk = calculate_total_coordination_risk(schedule_df, self.G, self.dependencies)
        imbalance = calculate_workload_imbalance(schedule_df, self.developers)

        score = makespan + total_risk + imbalance
        return score, {"makespan": makespan, "cr": total_risk, "imbalance": imbalance}

    def optimize(self):
        """
        Executes the hybrid PSO-GA optimization loop incorporating continuous PSO updates,
        blend crossover, and continuous mutation.
        """
        swarm = self.initialize_swarm()

        for _ in range(self.max_iter):
            for particle in swarm:
                schedule = self.decode_schedule(particle["position"])
                score, _ = self.evaluate_objectives(schedule)

                if score < particle["pbest_score"]:
                    particle["pbest_score"] = score
                    particle["pbest_position"] = particle["position"].copy()
                    particle["pbest_schedule"] = schedule

                if score < self.global_best_score:
                    self.global_best_score = score
                    self.global_best_position = particle["position"].copy()
                    self.global_best_schedule = schedule

            w, c1, c2 = 0.5, 1.5, 1.5
            new_swarm = []

            for particle in swarm:
                r1, r2 = np.random.rand(), np.random.rand()

                velocity = (
                    w * particle["velocity"] +
                    c1 * r1 * (particle["pbest_position"] - particle["position"]) +
                    c2 * r2 * (self.global_best_position - particle["position"])
                )

                position = particle["position"] + velocity
                position = np.clip(position, 0.0, 1.0)

                if self.global_best_position is not None:
                    if random.random() < 0.8:
                        position = self.crossover(position, self.global_best_position)

                position = self.mutate(position, mutation_rate=0.15)

                new_particle = {
                    "position": position,
                    "velocity": velocity,
                    "pbest_position": particle["pbest_position"],
                    "pbest_score": particle["pbest_score"],
                    "pbest_schedule": particle["pbest_schedule"]
                }
                new_swarm.append(new_particle)

            swarm = new_swarm

        return self.global_best_schedule

### IV.3 Test-Runner

In [ ]:
def run_and_evaluate_scheduler(
        solver_name: str, 
        tasks: DataFrame, 
        developers: DataFrame, 
        dependencies: list[tuple[str, str, dict[str, str]]], 
        horizon: int = None, 
        priority_order: dict = None, 
        w_makespan: float = None, 
        w_risk: float = None, 
        w_imbalance: float = None, 
        swarm_size:int = None, 
        max_iter: int = None
) -> DataFrame:
    """
    Unified execution function for different schedulers (CP-SAT or PSO-GA).
    Returns the schedule and prints analysis metrics. \\
    Any arguments left set as None would default to the values as per the chosen scheduler parameterization.
    """
    
    # 1. Execute the chosen solver
    if solver_name.lower() in ['cp-sat-lx']:
        kwargs = {
            'horizon': horizon,
            'priority_order': priority_order
        }
        kwargs = {k: v for k, v in kwargs.items() if v is not None}
        
        raw_schedule = cp_sat_lexicographic_scheduler(
            tasks, 
            developers, 
            dependencies,
            **kwargs  # Unpacks only the parameters that were explicitly provided
        )
    elif solver_name.lower() in ['cp-sat-ws']:
        kwargs = {
            'horizon': horizon,
            'w_makespan': w_makespan,
            'w_risk': w_risk,
            'w_imbalance': w_imbalance
        }
        kwargs = {k: v for k, v in kwargs.items() if v is not None}
        
        raw_schedule = cp_sat_weighted_sum_scheduler(
            tasks, 
            developers, 
            dependencies,
            **kwargs  # Unpacks only the parameters that were explicitly provided
        )
    elif solver_name.lower() in ['pso-ga', 'hybrid', 'metaheuristic']:
        kwargs = {
            'horizon': horizon,
            'swarm_size': swarm_size,
            'max_iter': max_iter
        }
        kwargs = {k: v for k, v in kwargs.items() if v is not None}
        scheduler = HybridSchedulerPSOGA(
            tasks, 
            developers,
            dependencies, 
            **kwargs  # Unpacks only the parameters that were explicitly provided
        )
        raw_schedule = scheduler.optimize()
        print("Optimization Finished Successfully!")
    else:
        raise ValueError(f"Unknown solver name: {solver_name}. Choose 'cp-sat' or 'pso-ga'.")

    # 2. Convert to DataFrame
    if raw_schedule:
        schedule_df = DataFrame.from_dict(
            raw_schedule,
            orient='index',
            columns=['assigned_dev', 'start', 'finish']
        )
    else:
        return
    
    print(f"\n--- Schedule Result ({solver_name.upper()}) ---")
    display(schedule_df)
    display_gantt_chart(schedule_df)

    # 3. Calculate Metrics
    makespan = calculate_makespan(schedule_df)
    print(f"Makespan: {makespan}")

    total_risk = calculate_total_coordination_risk(schedule_df, build_dependency_graph(tasks, dependencies), dependencies)
    print(f"Total Coordination Risk: {total_risk}")

    imbalance = calculate_workload_imbalance(schedule_df, developers)
    print(f"Workload Imbalance: {imbalance}")

    is_valid = check_constraints(schedule_df, tasks, developers, dependencies)
    print(f"Is schedule valid? {is_valid}\n")

    return schedule_df

## V - Running the schedulers & Displaying the results

In [ ]:
initialize(10, 0.8, 1.2, 5)

tasks, developers, dependencies = generate_project_data(
    20,
    8,
    1000,
    toggle_display=True
)
#schedule_cpsat_ws = run_and_evaluate_scheduler('cp-sat-ws', tasks, developers, dependencies, w_makespan=1, w_risk=1, w_imbalance=1)
schedule_cpsat_lx = run_and_evaluate_scheduler('cp-sat-lx', tasks, developers, dependencies, priority_order=['makespan', 'imbalance', 'total_risk'])
#schedule_psoga = run_and_evaluate_scheduler('pso-ga', tasks, developers, dependencies, swarm_size=20, max_iter=10)